# AstroVIPER Spectral Moment Tutorial

In [ ]:
from toolviper.utils.data import download, list_files

download(file="Antennae_North.cal.lsrk.ps.zarr")

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

# import dask
# dask.config.set(scheduler='synchronous')

In [ ]:
ps_store = "Antennae_North.cal.lsrk.ps.zarr"

from xradio.measurement_set import open_processing_set

scan_intents = ["OBSERVE_TARGET#ON_SOURCE"]
ps = open_processing_set(ps_store, scan_intents=scan_intents)
ps.xr_ps.summary()

In [ ]:
ps.xr_ps.plot_phase_centers()

In [ ]:
import numpy as np
import xarray as xr
import dask
from graphviper.graph_tools.coordinate_utils import (
    make_parallel_coord,
    make_frequency_coord,
)
from graphviper.graph_tools import map, reduce
from astroviper.core.image_analysis.moment_max import moment_max
from astroviper.distributed.imaging.utils import make_image_mosaic
from toolviper.utils.display import dict_to_html
from IPython.display import HTML, display

ps_store = "Antennae_North.cal.lsrk.ps.zarr"
scan_intents = ["OBSERVE_TARGET#ON_SOURCE"]
ps = open_processing_set(ps_store, scan_intents=scan_intents)

grid_params = {}
grid_params["chan_mode"] = "cube"
grid_params["image_size"] = [500, 500]
grid_params["cell_size"] = np.array([-0.13, 0.13]) * np.pi / (180 * 3600)
grid_params["fft_padding"] = 1.0

combined_field_and_source_xds = ps.xr_ps.get_combined_field_and_source_xds()
center_field_name = combined_field_and_source_xds.attrs["center_field_name"]
grid_params["phase_direction"] = (
    combined_field_and_source_xds.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=center_field_name
    )
)

input_params = {}
input_params["grid_params"] = grid_params
input_params["to_disk"] = False
input_params["input_data_store"] = ps_store
input_params["polarization"] = ["XX", "YY"]
input_params["time"] = [0]
input_params["frequency"] = None
input_params["data_group"] = "base"

parallel_coords = {}
# coord = make_frequency_coord(
#     freq_start=343018346078.4616,
#     freq_delta=11231488.981445312,
#     n_channels=166,
# )
# coord = ps['Antennae_North.cal.lsrk.split_04'].frequency
coord = ps["Antennae_North.cal.lsrk_34"].frequency

parallel_coords["frequency"] = make_parallel_coord(coord=coord, n_chunks=20)
from graphviper.graph_tools.coordinate_utils import (
    interpolate_data_coords_onto_parallel_coords,
)

node_task_data_mapping = interpolate_data_coords_onto_parallel_coords(
    parallel_coords, ps
)
display(HTML(dict_to_html(node_task_data_mapping)))

import time

graph = map(
    input_data=ps,
    node_task_data_mapping=node_task_data_mapping,
    node_task=make_image_mosaic,
    input_params=input_params,
    in_memory_compute=False,
)

input_params = {}
viper_graph = reduce(graph, moment_max, input_params)

from graphviper.graph_tools import generate_dask_workflow

dask_graph = generate_dask_workflow(viper_graph)

In [ ]:
dask.visualize(dask_graph)

In [ ]:
img_xds = dask.compute(dask_graph)[0]
img_xds

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
img_xds.SKY.isel(polarization=0, frequency=0, dummy=0).plot.imshow(
    cmap="viridis", vmin=0.0
)
plt.figure()
img_xds.PRIMARY_BEAM.isel(polarization=0, frequency=0, dummy=0).plot()

In [ ]:
img_xds.to_zarr("Antennae_North_MOM8.img.zarr", mode="w")